# Gating threshold study: experiments

This notebook finds where to set the welfare gates on the **Personalized** recommender, the study's Standard RS. It runs on the lab server, which holds
the database and the metro networks, and writes small result files to `results/`. Those files are all
that `analysis.ipynb` needs, so the analysis can run anywhere.

**Gates** (after Uğurel & Yabe 2026)
* **PUP** passes a venue only if $\Pr[U \ge 0] \ge \alpha$. Swept over $\alpha = 0.1, 0.2, \dots, 1.0$.
* **RM** passes a venue only if its regret $\hat R_k = \max_j \hat U_j - \hat U_k \le \varepsilon$: how far its estimated net utility $\hat U = \hat V - \hat C$
  falls short of the best candidate in the same recommendation list (the paper's Eq. 18). Swept over $\varepsilon = 0$ (only the best-estimated candidate
  passes) and $\varepsilon$ set at the 60th–95th percentiles of the regret of the ungated recommender's suggestions, as in the paper.

Each (city, seed, threshold) gets one PUP run and one RM run. The controls (No RS, Standard RS = ungated Personalized, and Oracle) run once
per (city, seed) and appear as flat reference lines on the threshold graphs. Size: 8 metros, 2,000 agents, 30 days, 5 seeds.
Oracle is optional (`RUN_ORACLE`): it does not affect the choice of threshold and is by far the slowest arm.

**Order of work.** The RM grid is defined by Standard RS's regrets, so Standard has to finish before the RM sweep starts.
1. Standard RS and No RS, then calibrate the RM grid from Standard RS.
2. The PUP sweep and the RM sweep, in one pool.
3. Oracle, the upper-bound reference, when `RUN_ORACLE` is set. It does not affect the choice of threshold and is by far the slowest arm, so it runs last.
4. Export one CSV per city, then write `results/DONE.txt`.

Each finished simulation is cached under `.cache/gating_study/`. An interrupted notebook (for example after a reboot) resumes from where it
stopped. The cache cannot tell that the model has changed, so delete it after any change to the simulation. Run it unattended with `./run_in_background.sh`; progress is appended to `results/progress.log`.

## Setup

In [1]:
import gc
import json
import multiprocessing as mp
import os
import pickle
import time
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

REPO = Path.cwd().parent

# The server datasource (Postgres: POIs, ACS/PUMS populations, survey personas) is configured in .env.server.
for line in (REPO / ".env.server").read_text().splitlines():
    if line.strip() and not line.lstrip().startswith("#") and "=" in line:
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"'))

from welfare_rs import TravelCostEstimator, WelfareAwareOrchestrator, metrics, params
from welfare_rs.datasource import get_datasource
from welfare_rs.experiment_harness import table1_metrics
from welfare_rs.metro import build_network
from welfare_rs.oracle_rs import oracle_factory
from welfare_rs.recommender_systems import RECOMMENDER_PRESETS, RecommenderConfig
from welfare_rs.simulation import Simulation

In [2]:
# Experiment size
CITIES = ["houston", "la", "chicago", "sf_bay", "nyc", "dc", "seattle", "miami"]
N_AGENTS = 2000
N_DAYS = 30
SEEDS = list(range(5))

# Threshold grids (the paper's ranges)
PUP_ALPHAS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
# RM: p0 is epsilon = 0 exactly (the best candidate in every list has zero regret); it replaces p50.
RM_PERCENTILES = [0, 60, 70, 75, 80, 90, 95]
# The Oracle upper bound is a reference line only; it does not change which threshold is best.
RUN_ORACLE = False

# The recommender whose gates are tuned: the study's Standard RS.
RECOMMENDER = "Personalized"
BASE_RS = RecommenderConfig(label=RECOMMENDER, **RECOMMENDER_PRESETS[RECOMMENDER])

# Concurrency: up to MAX_PARALLEL runs, one per core. A run starts only if the runs' memory estimates fit in
# the budget and the box still has MIN_AVAILABLE_GB free. Estimates are peak private memory of one run while
# sharing the network, with margin (measured up to 2.2 GB for the other arms and 2.8 GB for Oracle on this model).
MAX_PARALLEL = 62
MEMORY_BUDGET_GB = 180
RUN_MEMORY_GB = {"oracle": 4.0}
DEFAULT_RUN_MEMORY_GB = 3.0
MIN_AVAILABLE_GB = 30

RESULTS_DIR = Path("results")
CACHE_DIR = REPO / ".cache" / "gating_study" / f"{RECOMMENDER.lower()}_{N_AGENTS}agents_{N_DAYS}days"
RESULTS_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"{len(CITIES)} cities x {len(SEEDS)} seeds, {N_AGENTS} agents, {N_DAYS} days")

8 cities x 5 seeds, 2000 agents, 30 days


## One shared copy of each metro network

A network is 0.7–3 GB. It is loaded once here, and every run is a forked copy that shares it instead of loading its own, which halves each
run's memory and is what lets one run go on every core.

In [3]:
NETWORKS = {city: build_network(city) for city in CITIES}
gc.collect()
gc.freeze()  # keep the garbage collector from touching (and so copying) the shared networks in every run

## Arms and the single-run function

An `Arm` is one recommender condition. Gated arms wrap the Personalized recommender in the engine's `WelfareAwareOrchestrator`,
wired exactly as the web app wires a gated recommender (`backend/service.py`).

Standard RS goes through the same wrapper in pass-through mode (`"standard"`). It records the regret of every candidate,
which is used to calibrate the RM grid. The pass-through filters nothing and draws no random numbers, so the run is identical to an
unwrapped Personalized recommender.

In [4]:
@dataclass(frozen=True)
class Arm:
    gate: str                           # "no_rs", "oracle", "standard", "pup" or "rm"
    alpha: Optional[float] = None       # PUP: pass a venue only if P(U >= 0) >= alpha
    epsilon: Optional[float] = None     # RM:  pass a venue only if regret <= epsilon
    percentile: Optional[int] = None    # RM:  the regret percentile epsilon was calibrated at

    @property
    def key(self):
        parts = [self.gate]
        if self.alpha is not None:
            parts.append(f"a{self.alpha:.2f}")
        if self.epsilon is not None:
            parts.append(f"e{self.epsilon:.4f}")
        return "_".join(parts)


class RegretLoggingGate(WelfareAwareOrchestrator):
    # The engine's welfare gate, also recording the regret of every candidate it passes.
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.regrets = []

    def recommend(self, user, leisure_subtype, top_k_per_system=5):
        passed = super().recommend(user, leisure_subtype, top_k_per_system)
        self.regrets.extend(rec.regret for recs in passed.values() for rec in recs)
        return passed


def recommender_factory(arm):
    if arm.gate == "no_rs":
        return None
    if arm.gate == "oracle":
        return oracle_factory("Oracle")
    gate = {"mode": "standard" if arm.gate == "standard" else arm.gate}
    if arm.alpha is not None:
        gate["pup_alpha"] = arm.alpha
    if arm.epsilon is not None:
        gate["rm_epsilon"] = arm.epsilon

    def factory(sim, base_stack):
        # The metro sets the taxi tariff in the gate's travel-cost estimate, as in the web app.
        return RegretLoggingGate(base_stack, TravelCostEstimator(metro=sim.metro), **gate)
    return factory


def run_one(city, seed, arm):
    # One simulation. Population and network setup mirror the web app's study runner.
    ds = get_datasource()
    population = ds.population(city, replicate=seed % ds.population_replicates(city),
                               limit=N_AGENTS + max(100, N_AGENTS // 5))
    if population is None or len(population) < N_AGENTS:
        raise RuntimeError(f"{city}: no built population with {N_AGENTS} agents")
    sim = Simulation(
        num_agents=N_AGENTS, seed=seed,
        use_recommenders=(arm.gate != "no_rs"),
        persona_rows=ds.personas(),
        persona_csv_path=ds.persona_csv_path() or params.SURVEY_PERSONA_CSV_PATH,
        population_rows=population,
        road_network=NETWORKS[city],
        poi_rows=ds.poi_rows(city),
        disabled_modes=("transit",),
        recommender_config=BASE_RS,
        recommender_factory=recommender_factory(arm),
    )
    sim.run_days(N_DAYS)
    table1 = table1_metrics(sim)  # paper Table 1, over the evaluation (last) day's leisure trips
    taste = metrics.taste_metrics(sim)  # the same day's leisure visits that matched the agent's latent taste
    return {
        "city": city, "seed": seed, "arm": arm.key, "gate": arm.gate,
        "alpha": arm.alpha, "percentile": arm.percentile, "epsilon": arm.epsilon,
        "mean_utility": table1["mean_utility"], "abstention_rate": table1["abstention_rate"],
        "taste_match": taste["all_rate"], "taste_match_recommended": taste["recommended_rate"],
        "taste_match_organic": taste["organic_rate"],
        "regrets": np.asarray(sim.recommender_stack.regrets, dtype=np.float32) if arm.gate == "standard" else None,
    }

## Parallel runner

Each `(city, seed, arm)` simulation runs in its own process, forked from this notebook so that it shares the networks. When a run finishes,
its process exits and the memory goes back to the system. A run is started whenever a core is free and its memory estimate fits.
A failed run is logged and does not stop the others; re-running the notebook retries only the runs that are missing.

In [5]:
def log(message):
    line = f"{datetime.now():%Y-%m-%d %H:%M}  {message}"
    print(line, flush=True)
    with open(RESULTS_DIR / "progress.log", "a") as f:
        f.write(line + "\n")


def available_gb():
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith("MemAvailable"):
                return int(line.split()[1]) / 1e6


def run_memory_gb(task):
    return RUN_MEMORY_GB.get(task[2].gate, DEFAULT_RUN_MEMORY_GB)


def can_start(task, running):
    reserved = sum(run_memory_gb(other) for other in running.values())
    return (len(running) < MAX_PARALLEL and reserved + run_memory_gb(task) <= MEMORY_BUDGET_GB
            and available_gb() >= MIN_AVAILABLE_GB)


def cache_path(city, seed, arm):
    return CACHE_DIR / f"{city}_seed{seed}_{arm.key}.pkl"


def run_grid(arms):
    # Run every (city, seed, arm) not yet cached, in the order of `arms`, and return all of them as one row per run.
    tasks = [(city, seed, arm) for arm in arms for city in CITIES for seed in SEEDS]
    pending = [task for task in tasks if not cache_path(*task).exists()]
    n_todo = len(pending)
    log(f"{len(tasks) - n_todo} of {len(tasks)} runs cached; running {n_todo}")

    running, failed, finished = {}, [], 0
    # maxtasksperchild=1: a fresh process per run. fork: runs share the networks and the functions defined here.
    with mp.get_context("fork").Pool(MAX_PARALLEL, maxtasksperchild=1) as pool:
        while pending or running:
            # Start pending runs while cores and memory allow.
            while pending and can_start(pending[0], running):
                task = pending.pop(0)
                running[pool.apply_async(run_one, task)] = task
            time.sleep(10)
            for job in [job for job in running if job.ready()]:
                city, seed, arm = task = running.pop(job)
                finished += 1
                try:
                    result = job.get()
                except Exception as exc:
                    failed.append(task)
                    log(f"[{finished}/{n_todo}] {city} seed {seed} {arm.key}  FAILED: {exc!r}")
                    continue
                path = cache_path(*task)
                tmp = path.with_suffix(".tmp")
                tmp.write_bytes(pickle.dumps(result))
                tmp.replace(path)  # atomic: an interrupted write never leaves a half-written cache file
                log(f"[{finished}/{n_todo}] {city} seed {seed} {arm.key}  U = {result['mean_utility']:+.3f}  "
                    f"abstention = {result['abstention_rate']:.2f}")
    if failed:
        raise RuntimeError(f"{len(failed)} runs failed (see results/progress.log); re-run to retry them")
    return pd.DataFrame([pickle.loads(cache_path(*task).read_bytes()) for task in tasks])

## Phase 1: Standard RS and No RS

In [6]:
NO_RS, STANDARD, ORACLE = Arm("no_rs"), Arm("standard"), Arm("oracle")
phase1 = run_grid([STANDARD, NO_RS])

2026-10-19 14:47  0 of 80 runs cached; running 80


2026-10-19 14:57  [1/80] houston seed 2 no_rs  U = +0.149  abstention = 1.00


2026-10-19 14:57  [2/80] houston seed 0 no_rs  U = +0.148  abstention = 1.00


2026-10-19 14:57  [3/80] houston seed 1 no_rs  U = +0.135  abstention = 1.00


2026-10-19 14:57  [4/80] houston seed 4 no_rs  U = +0.149  abstention = 1.00


...

2026-10-19 15:43  [76/80] la seed 0 standard  U = +0.328  abstention = 0.00


2026-10-19 15:43  [77/80] la seed 1 standard  U = +0.308  abstention = 0.00


2026-10-19 15:43  [78/80] la seed 4 standard  U = +0.330  abstention = 0.00


2026-10-19 15:44  [79/80] la seed 2 standard  U = +0.312  abstention = 0.00


2026-10-19 15:45  [80/80] la seed 3 standard  U = +0.328  abstention = 0.00


## RM calibration

RM thresholds have no natural scale, so the paper sets $\varepsilon$ at percentiles of the regret that Standard RS's suggestions carry. The percentiles
are taken over every candidate it recommended, in every city, seed and day, which gives one $\varepsilon$ grid shared by all cities. A lower percentile
is a stricter gate.

In [7]:
regrets = np.concatenate(phase1.loc[phase1["gate"] == "standard", "regrets"].to_list())
RM_EPSILONS = {p: round(float(np.percentile(regrets, p)), 4) for p in RM_PERCENTILES}
(RESULTS_DIR / "rm_epsilons.json").write_text(json.dumps(
    {"n_recommendations": len(regrets), "epsilon_by_percentile": RM_EPSILONS}, indent=2))

print(f"{len(regrets):,} Standard-RS recommendations")
pd.Series(RM_EPSILONS, name="epsilon").rename_axis("percentile").to_frame()

4,108,683 Standard-RS recommendations


,epsilon
percentile,
0,0.0000
60,0.0435
70,0.0640
75,0.0782
80,0.0976
90,0.1701
95,0.2700


## Phase 2: the PUP and RM sweeps

In [8]:
PUP_ARMS = [Arm("pup", alpha=alpha) for alpha in PUP_ALPHAS]
RM_ARMS = [Arm("rm", epsilon=eps, percentile=p) for p, eps in RM_EPSILONS.items()]
phase2 = run_grid([*PUP_ARMS, *RM_ARMS])

2026-10-19 15:45  0 of 680 runs cached; running 680


2026-10-19 16:19  [1/680] miami seed 4 pup_a0.10  U = +0.358  abstention = 0.00


2026-10-19 16:19  [2/680] miami seed 1 pup_a0.10  U = +0.352  abstention = 0.00


2026-10-19 16:19  [3/680] miami seed 2 pup_a0.10  U = +0.349  abstention = 0.00


2026-10-19 16:19  [4/680] miami seed 0 pup_a0.10  U = +0.362  abstention = 0.00


...

2026-10-20 01:39  [676/680] seattle seed 4 rm_e0.2700  U = +0.285  abstention = 0.00


2026-10-20 01:44  [677/680] nyc seed 2 rm_e0.2700  U = +0.253  abstention = 0.00


2026-10-20 01:44  [678/680] nyc seed 1 rm_e0.2700  U = +0.265  abstention = 0.00


2026-10-20 01:45  [679/680] nyc seed 4 rm_e0.2700  U = +0.256  abstention = 0.00


2026-10-20 01:46  [680/680] nyc seed 3 rm_e0.2700  U = +0.246  abstention = 0.00


## Phase 3: Oracle

The perfect-knowledge upper bound: ranks every venue by the agent's true net utility. A reference line on the graphs only, run when `RUN_ORACLE` is set.

In [9]:
phase3 = run_grid([ORACLE]) if RUN_ORACLE else pd.DataFrame()

## Export the results

One CSV per city with one row per run, holding only what the threshold graphs need: the arm and its threshold, plus the run's
evaluation-day mean net utility and abstention rate (paper Table 1 definitions), and the share of that day's leisure visits
that matched the agent's taste (all, recommended and organic). `config.json` records the design. `DONE.txt` is written
last, so it appears only once every result file is complete.

In [10]:
runs = pd.concat([phase1, phase2, phase3], ignore_index=True)
columns = ["city", "seed", "arm", "gate", "alpha", "percentile", "epsilon", "mean_utility", "abstention_rate",
           "taste_match", "taste_match_recommended", "taste_match_organic"]
for city, city_runs in runs.groupby("city"):
    city_runs = city_runs[columns].astype({"percentile": "Int64"})  # integer percentiles, blank where not RM
    city_runs.sort_values(["arm", "seed"]).to_csv(RESULTS_DIR / f"{city}.csv", index=False)

(RESULTS_DIR / "config.json").write_text(json.dumps({
    "cities": CITIES, "n_agents": N_AGENTS, "n_days": N_DAYS, "seeds": SEEDS,
    "recommender": RECOMMENDER, "pup_alphas": PUP_ALPHAS, "rm_percentiles": RM_PERCENTILES,
    "oracle": RUN_ORACLE, "surprise_weight": params.FEEDBACK_PARAMS["surprise_weight"],
}, indent=2))

(RESULTS_DIR / "DONE.txt").write_text(
    f"All experiments finished {datetime.now():%Y-%m-%d %H:%M}.\n"
    f"{len(runs)} runs: {len(CITIES)} cities x {len(SEEDS)} seeds x {runs['arm'].nunique()} arms, "
    f"{N_AGENTS} agents, {N_DAYS} days.\n"
)
log(f"DONE: {len(runs)} runs exported to {RESULTS_DIR.resolve()}")

/tmp/ipykernel_1975543/3978974292.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  runs = pd.concat([phase1, phase2, phase3], ignore_index=True)


2026-10-20 01:46  DONE: 760 runs exported to /home/airecsim/welfare_recsys_viz/gating_study/results
